**ResNet**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


Dataset Preparation

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

# Global transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
]) ## This section written by gemini

def get_dataloaders(split_name, batch_size=64):
    train_dataset = torchvision.datasets.EMNIST(
        root='./data', train=True, split=split_name, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.EMNIST(
        root='./data', train=False, split=split_name, download=True, transform=transform
    )
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Determine number of classes from the dataset labels
    num_classes = len(train_dataset.classes)
    return train_loader, test_loader, num_classes

Final Model Definition



In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out
## Residual Block code written with assistance from Gemini

class ResidualCNN(nn.Module):
    def __init__(self, num_classes):
        super(ResidualCNN, self).__init__()
        self.in_channels = 16

        # Initial layer
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # ResNet Layers
        self.layer1 = self._make_layer(16, 2, stride=1)
        self.layer2 = self._make_layer(32, 2, stride=2)
        self.layer3 = self._make_layer(64, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride): ## This function written with help from Gemini, what I had previously was very broken
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [4]:
import torch.optim as optim
import os

def train_evaluate_save(split_name, epochs=20):
    print(f"\n--- Processing EMNIST split: {split_name} ---")
    save_base_dir = '/content/drive/MyDrive/cs320/final project/models'
    os.makedirs(save_base_dir, exist_ok=True) # written by gemini

    train_loader, test_loader, num_classes = get_dataloaders(split_name)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ResidualCNN(num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1): # Start epoch from 1 for saving logic
        model.train()
        total_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[{split_name}] Epoch {epoch} Avg Loss: {total_loss/len(train_loader):.4f}")

        # Save model at specific epochs to Google Drive
        if epoch == 5 or epoch == 10 or epoch == 20:
            save_path_epoch = os.path.join(save_base_dir, f"resnet_{split_name}_epoch_{epoch}.pth")
            torch.save(model.state_dict(), save_path_epoch)
            print(f"Model saved to {save_path_epoch} at epoch {epoch}")
            ## Saving section written by gemini

    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Accuracy on {split_name}: {accuracy:.2f}%")

    # Save Final State to Google Drive
    final_save_path = os.path.join(save_base_dir, f"resnet_{split_name}_final.pth")
    torch.save(model.state_dict(), final_save_path)
    print(f"Final model saved to {final_save_path}")
    return accuracy

# Execute for each split
splits = ['bymerge', 'byclass', 'balanced']
results = {}

for s in splits:
    results[s] = train_evaluate_save(s, epochs=20)

print("\nSummary of Results:")
for split, acc in results.items():
    print(f"{split}: {acc:.2f}%")


--- Processing EMNIST split: bymerge ---


100%|██████████| 562M/562M [00:04<00:00, 124MB/s]


KeyboardInterrupt: 

## 5. Model Evaluation: Accuracy, Precision, Recall, and F1-Score

We will now load the final trained models for each EMNIST split (`bymerge`, `byclass`, `balanced`) and evaluate their performance using a comprehensive set of metrics: accuracy, precision, recall, and F1-score. These metrics provide a more detailed understanding of the model's classification capabilities beyond simple accuracy, especially in multi-class scenarios.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import os

def evaluate_model_comprehensive(model, test_loader, device, split_name):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    print(f"Metrics for {split_name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}\n")
    ## Gemini wrote this print formatting

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

KeyboardInterrupt: 

In [ ]:
save_base_dir = '/content/drive/MyDrive/cs320/final project/models'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits_to_eval = ['bymerge','byclass', 'balanced']
final_results = {}

for split in splits_to_eval:
    print(f"--- Evaluating final model for split: {split} ---")
    train_loader, test_loader, num_classes = get_dataloaders(split)

    # Initialize model and load weights
    model = ResidualCNN(num_classes).to(device)
    model_path = os.path.join(save_base_dir, f"resnet_{split}_final.pth")

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        final_results[split] = evaluate_model_comprehensive(model, test_loader, device, split)
    else:
        print(f"Model file not found at {model_path}")
        print(f"  F1-Score: {metrics['f1_score']:.4f}")


Custom Dataset

In [15]:
import os
import torch
from PIL import Image, ImageOps
import torchvision.transforms as transforms

# --- 1. Label Mappings ---

# Map your custom file numbers to the actual lowercase characters
custom_to_char = {
    1: 'q', 2: 'l', 3: 'p', 4: 'o', 5: 'w', 6: 't', 7: 'v', 8: 'n',
    9: 'h', 10: 'g', 11: 'f', 12: 'd', 13: 'k', 14: 'a', 15: 'c',
    16: 'u', 17: 'j', 18: 'x', 19: 'i', 20: 's', 21: 'e', 22: 'r',
    23: 'b', 24: 'z', 25: 'y', 26: 'm'
}

# Map characters to EMNIST 'byclass' indices (lowercase 'a' is 36, 'z' is 61)
char_to_emnist_byclass = {chr(i + 97): i + 36 for i in range(26)}

def preprocess_custom_image(image_path, is_dark_text=True, fix_emnist_rotation=True):
    # 1. Load image in grayscale
    img = Image.open(image_path).convert('L')

    # 2. Invert so text is white on black background
    if is_dark_text:
        img = ImageOps.invert(img)

    # 3. THRESHOLDING: Force dark greys (shadows/paper) to pure black and text to pure white
    # Anything below the pixel value of 100 becomes 0. You can tweak this number (e.g., 80 to 150)
    # if your lighting was particularly bright or dark!
    img = img.point(lambda p: 255 if p > 100 else 0)

    # 4. AUTO-CROPPING: Remove empty black space so the letter fills the frame like EMNIST
    bbox = img.getbbox() # Gets bounding box of the non-black pixels
    if bbox:
        img = img.crop(bbox)

    # 5. Padding & Resizing: Add a tiny 15% border (standard for EMNIST), then resize to 32x32
    padding = max(img.size) // 6
    img = ImageOps.expand(img, border=padding, fill=0)
    img = img.resize((32, 32), Image.Resampling.BILINEAR)

    # 6. Fix EMNIST Rotation Quirk
    if fix_emnist_rotation:
        img = img.rotate(-90, expand=True)
        img = ImageOps.mirror(img)

    # 7. Convert to tensor and apply your training normalization
    transform_pipeline = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    return transform_pipeline(img).unsqueeze(0)

# --- Full EMNIST Decoder Dictionary ---
emnist_decoder = {}
for i in range(10): emnist_decoder[i] = str(i)             # 0-9
for i in range(26): emnist_decoder[i + 10] = chr(i + 65)   # A-Z
for i in range(26): emnist_decoder[i + 36] = chr(i + 97)   # a-z

# --- 3. Evaluation Loop ---
def evaluate_custom_dataset(model, folder_path, device):
    model.eval()

    results = {
        'total': {'strict_correct': 0, 'case_insensitive_correct': 0, 'count': 0},
        'messy': {'strict_correct': 0, 'case_insensitive_correct': 0, 'count': 0},
        'print': {'strict_correct': 0, 'case_insensitive_correct': 0, 'count': 0}
    }

    print(f"Evaluating images in: {folder_path}...\n")

    with torch.no_grad():
        for filename in os.listdir(folder_path):
            if not filename.endswith(('.png', '.jpg', '.jpeg')):
                continue

            parts = filename.replace('.png', '').replace('.jpg', '').split('_')
            if len(parts) != 3:
                continue

            writing_type = parts[1]
            custom_class = int(parts[2])

            true_char_lower = custom_to_char[custom_class]
            true_emnist_label = char_to_emnist_byclass[true_char_lower]

            img_path = os.path.join(folder_path, filename)
            img_tensor = preprocess_custom_image(img_path).to(device)

            outputs = model(img_tensor)
            _, predicted_label = torch.max(outputs.data, 1)
            predicted_label = predicted_label.item()

            # Use our new decoder to get the actual predicted letter!
            pred_char = emnist_decoder[predicted_label]

            # --- Check Accuracy ---
            # Strict: Must match exactly (e.g., lowercase 'w' == lowercase 'w')
            strict_match = (predicted_label == true_emnist_label)

            # Case-Insensitive: Matches if letters are the same, regardless of case
            case_match = (pred_char.lower() == true_char_lower.lower())

            # Update stats
            results['total']['count'] += 1
            results[writing_type]['count'] += 1

            if strict_match:
                results['total']['strict_correct'] += 1
                results[writing_type]['strict_correct'] += 1
            if case_match:
                results['total']['case_insensitive_correct'] += 1
                results[writing_type]['case_insensitive_correct'] += 1

            # Print failures (only print if it failed the case-insensitive check)
            if not case_match:
                print(f"True Mistake on {filename}: True={true_char_lower}, Predicted={pred_char}")

    # --- Print Summary ---
    print("\n--- Custom Dataset Results ---")
    for category in ['total', 'messy', 'print']:
        count = results[category]['count']
        if count > 0:
            strict_acc = results[category]['strict_correct'] / count * 100
            case_acc = results[category]['case_insensitive_correct'] / count * 100
            print(f"{category.capitalize()} - Strict Accuracy: {strict_acc:.2f}% | Case-Insensitive Accuracy: {case_acc:.2f}%")

# --- 4. Execution ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_classes = 62
model = ResidualCNN(num_classes).to(device)

# 3. Load the Weights
model_path = "/content/drive/MyDrive/cs320/final project/models/resnet_byclass_final.pth"

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Successfully loaded model weights from {model_path}")

    folder_dir = "/content/drive/MyDrive/cs320/final project/handwriting_data"

    if os.path.exists(folder_dir):
        evaluate_custom_dataset(model, folder_dir, device)
    else:
        print(f"Error: Could not find image folder at {folder_dir}. Check the path.")
else:
    print(f"Error: Could not find model file at {model_path}. Check the path.")

Using device: cuda
Successfully loaded model weights from /content/drive/MyDrive/cs320/final project/models/resnet_byclass_final.pth
Evaluating images in: /content/drive/MyDrive/cs320/final project/handwriting_data...

True Mistake on sample5_messy_1.png: True=q, Predicted=G
True Mistake on sample4_messy_9.png: True=h, Predicted=a
True Mistake on sample4_print_22.png: True=r, Predicted=K
True Mistake on sample3_messy_2.png: True=l, Predicted=B
True Mistake on sample4_messy_3.png: True=p, Predicted=W
True Mistake on sample4_messy_19.png: True=i, Predicted=B
True Mistake on sample5_print_19.png: True=i, Predicted=Z
True Mistake on sample5_messy_20.png: True=s, Predicted=G
True Mistake on sample5_messy_22.png: True=r, Predicted=b
True Mistake on sample4_messy_1.png: True=q, Predicted=g
True Mistake on sample5_messy_8.png: True=n, Predicted=B
True Mistake on sample5_messy_17.png: True=j, Predicted=0
True Mistake on sample4_print_3.png: True=p, Predicted=a
True Mistake on sample5_messy_15.p